# 18. Architecture Ablation — Classical → Modern（spec §7.3, §17, §18）

## 学習目標
- 1要素ずつ（LayerNorm→RMSNorm→RoPE→SwiGLU→bias無し）変えた統制実験を読む
- **効果量を seed 変動（3 seeds）と比較**し、「改善」を軽々に断定しない規律を身につける
- 「どちらが良いか」だけでなく、何が変わり・計算コストに見合うか・断定できないことを述べる

## 統制条件（ceteris paribus）
同一コーパス（pilot 8M）・同一 validation（別文書）・同一パラメータ規模（SwiGLU幅=8d/3でGELU 4dに一致）・
同一トークン予算（~3.7M, <1epoch）・同一 optimizer・各3 seeds。独立変数はアーキテクチャの1要素のみ。

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"

In [2]:
from jp_llm_lab.visualization.comparison import ablation_val_loss_figure, ablation_table_rows
summary = load_json(ROOT / "experiments/comparisons/ablation_chain.json")
ablation_val_loss_figure(summary).show()

In [3]:
rows = ablation_table_rows(summary)
df = pd.DataFrame(rows)
seed_std = np.mean([summary["results"][c]["val_loss_std"] for c in summary["chain"]])
print(f"平均 seed 標準偏差（ノイズ床）= {seed_std:.4f} nats")
print(f"classical→modern の総変化 = {rows[-1]['delta_vs_base']:+.4f} nats")
df

平均 seed 標準偏差（ノイズ床）= 0.0447 nats
classical→modern の総変化 = -0.3267 nats


,step,n_params,val_loss,std,delta_vs_prev,delta_vs_base,top1_conf
0,classical,10183680,6.4258,0.0454,0.0000,0.0000,0.0493
1,rmsnorm,10179520,6.4330,0.0038,0.0071,0.0071,0.0492
2,rope,10015680,6.0932,0.0501,-0.3397,-0.3326,0.0640
3,swiglu,10079808,6.0229,0.0248,-0.0703,-0.4029,0.0685
4,modern,10059840,6.0991,0.0996,0.0763,-0.3267,0.0630


In [4]:
# 各ステップの変化を seed ノイズと並べて可視化（|Δ| が 2σ を超えるか）
steps = summary["chain"][1:]
deltas = [r["delta_vs_prev"] for r in rows[1:]]
fig = go.Figure(go.Bar(x=steps, y=deltas,
                       marker_color=["#d62728" if abs(d) > 2*seed_std else "#7f7f7f" for d in deltas]))
fig.add_hline(y=2*seed_std, line_dash="dot", line_color="#d62728", annotation_text="+2σ")
fig.add_hline(y=-2*seed_std, line_dash="dot", line_color="#2ca02c", annotation_text="−2σ")
fig.update_layout(title="1要素変更ごとの Δval loss（赤=|Δ|>2σ で有意の可能性）",
                  yaxis_title="Δ val loss vs 前ステップ", template="plotly_white", height=380)
fig.show()
print("灰色バー = seed 変動と区別できない（この規模・この予算では有意と言えない）")

灰色バー = seed 変動と区別できない（この規模・この予算では有意と言えない）


In [5]:
# パラメータ数と較正（top-1 confidence）も比較
fig = go.Figure()
fig.add_trace(go.Bar(x=summary["chain"], y=[summary["results"][c]["n_params"] for c in summary["chain"]],
                     name="params", marker_color="#1f77b4"))
fig.update_layout(title="各構成のパラメータ数（ほぼ一定＝容量ではなく構造の比較）",
                  yaxis_title="parameters", template="plotly_white", height=320)
fig.show()
for c in summary["chain"]:
    r = summary["results"][c]
    print(f'{c:10s} val {r["val_loss_mean"]:.3f}±{r["val_loss_std"]:.3f}  '
          f'top1_conf {r["top1_conf_mean"]:.3f}  entropy {r["entropy_mean"]:.2f}')

classical  val 6.426±0.045  top1_conf 0.049  entropy 6.75
rmsnorm    val 6.433±0.004  top1_conf 0.049  entropy 6.77
rope       val 6.093±0.050  top1_conf 0.064  entropy 6.63
swiglu     val 6.023±0.025  top1_conf 0.069  entropy 6.62
modern     val 6.099±0.100  top1_conf 0.063  entropy 6.65


## Observation / Interpretation / Caveat
- **Observation（実測）**: 変更ごとに効果量が大きく異なった。**RMSNorm は seed 変動内（Δ≈+0.007, σ≈0.045）で改善なし**。**RoPE は大きな改善（Δ≈−0.34 nats）で 2σ を大きく超える**。SwiGLU はさらに小改善（Δ≈−0.07, 有意）。**bias 無し（→Modern）は Δ≈+0.08 だが自身の σ≈0.10 内で判定不能**。classical→modern の総改善は約 −0.33 nats。
- **Interpretation**: 事前予想（「小規模では全変更が seed ノイズに埋もれる」）は**RoPE について明確に外れた** — この10M・<1epoch の設定でも RoPE の相対位置符号化は学習を実質的に助けた。RMSNorm は「改善」ではなく**同等性能での簡素化**（パラメータ削減・mean中心化除去）として価値がある。bias 無しは中立（大規模での正則化/速度利点は別途）。**改善のほぼ全ては RoPE 由来**であり、「Modern が一律に良い」という粗い物語は誤り。
- **Caveat**: 3 seeds は区間推定が粗く、特に bias 無しの符号は確定できない。<1epoch・単一コーパス・10M規模。RoPE の利点が長文脈外挿でさらに広がるかは未検証（follow-up）。速度差はこの短期 run では測定ノイズが大きく比較していない。この結論を Model L 規模へ外挿してはいけない（M4 で確認）。

## 確認問題
1. 「step 軸で改善」を主張する前に確認すべき軸は何か（NB12 と関連）。
2. Δloss が seed σ の範囲内のとき、正しい結論は「改善なし」か「判定不能」か。
3. RoPE の利点を顕在化させるには、どんな追加実験（独立変数）が必要か。

**次へ（M4）**: [19_scaling_analysis](19_scaling_analysis.ipynb) — S/M/L のスケーリング比較。